In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (20,10)

from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import pickle
import json

In [ ]:
np.random.seed(42)

n_samples = 2000

sign_types = ["banner", "channel_letter", "monument", "vinyl"]
materials = ["PVC", "Aluminum", "Acrylic"]
complexity_levels = ["low", "medium", "high"]

data = []

for i in range(n_samples):

    # dimensions
    width = np.random.randint(2, 50)
    height = np.random.randint(2, 20)

    area = width * height

    # categorical features
    sign_type = np.random.choice(sign_types)
    material = np.random.choice(materials)
    complexity = np.random.choice(complexity_levels)

    # binary features
    lighting = np.random.choice([0, 1])
    installation = np.random.choice([0, 1])

    # quantity
    quantity = np.random.randint(1, 5)

    # =====================================
    # REALISTIC SIGN PRICING LOGIC
    # =====================================

    # material pricing impact
    material_multiplier = {
        "PVC": 1.0,
        "Aluminum": 1.5,
        "Acrylic": 2.0
    }

    # sign type pricing impact
    type_multiplier = {
        "banner": 1.0,
        "vinyl": 1.3,
        "channel_letter": 2.5,
        "monument": 3.5
    }

    # base square-foot pricing
    base_price = area * 5

    # apply material multiplier
    base_price *= material_multiplier[material]

    # apply sign type multiplier
    base_price *= type_multiplier[sign_type]

    # lighting cost
    if lighting:
        base_price += 300

    # installation cost
    if installation:
        base_price += 500

    # complexity adjustment
    if complexity == "medium":
        base_price *= 1.10

    elif complexity == "high":
        base_price *= 1.20

    # quantity scaling
    base_price *= quantity

    # random business noise
    price = base_price + np.random.normal(0, 100)

    # prevent negative prices
    price = max(price, 100)

    data.append([
        width,
        height,
        area,
        sign_type,
        material,
        complexity,
        lighting,
        installation,
        quantity,
        round(price, 2)
    ])

# create dataframe
df = pd.DataFrame(data, columns=[
    "width",
    "height",
    "area",
    "sign_type",
    "material",
    "complexity",
    "lighting",
    "installation",
    "quantity",
    "price"
])

df.head()

,width,height,area,sign_type,material,complexity,lighting,installation,quantity,price
0,40,16,640,monument,PVC,low,0,1,3,35090.54
1,37,9,333,vinyl,Acrylic,medium,0,1,4,21154.72
2,45,7,315,channel_letter,PVC,low,1,1,2,9231.09
3,29,17,493,monument,Aluminum,medium,0,1,3,44416.47
4,45,4,180,banner,Acrylic,high,0,0,3,6277.33


In [ ]:
df.shape
df.describe()
df.isnull().sum()

width           0
height          0
area            0
sign_type       0
material        0
complexity      0
lighting        0
installation    0
quantity        0
price           0
dtype: int64

In [ ]:
df = df[df["area"] > 1]
df = df[df["price"] > 0]

In [ ]:
df["price_per_area"] = df["price"] / df["area"]
df.head()

,width,height,area,sign_type,material,complexity,lighting,installation,quantity,price,price_per_area
0,40,16,640,monument,PVC,low,0,1,3,35090.54,54.828969
1,37,9,333,vinyl,Acrylic,medium,0,1,4,21154.72,63.527688
2,45,7,315,channel_letter,PVC,low,1,1,2,9231.09,29.305048
3,29,17,493,monument,Aluminum,medium,0,1,3,44416.47,90.094260
4,45,4,180,banner,Acrylic,high,0,0,3,6277.33,34.874056


In [ ]:
def remove_pps_outliers(df):
    df_out = pd.DataFrame()

    for key, subdf in df.groupby("material"):
        m = np.mean(subdf.price_per_area)
        st = np.std(subdf.price_per_area)

        reduced_df = subdf[
            (subdf.price_per_area > (m - st)) &
            (subdf.price_per_area <= (m + st))
        ]

        df_out = pd.concat([df_out, reduced_df], ignore_index=True)

    return df_out


df = remove_pps_outliers(df)
df.shape

(1572, 11)

In [ ]:
dummies_material = pd.get_dummies(df.material, dtype=int)
dummies_sign = pd.get_dummies(df.sign_type, dtype=int)
dummies_complexity = pd.get_dummies(df.complexity, dtype=int)

df2 = pd.concat([
    df,
    dummies_material,
    dummies_sign,
    dummies_complexity
], axis=1)

df2.drop(["material", "sign_type", "complexity", "price_per_area"], axis=1, inplace=True)

df2.columns = df2.columns.astype(str)
df2.head()

,width,height,area,lighting,installation,quantity,price,Acrylic,Aluminum,PVC,banner,channel_letter,monument,vinyl,high,low,medium
0,37,9,333,0,1,4,21154.72,1,0,0,0,0,0,1,0,0,1
1,45,4,180,0,0,3,6277.33,1,0,0,1,0,0,0,1,0,0
2,27,3,81,1,0,4,6606.34,1,0,0,0,0,0,1,1,0,0
3,46,10,460,0,0,3,13612.43,1,0,0,1,0,0,0,0,1,0
4,38,16,608,0,1,3,47096.42,1,0,0,0,1,0,0,0,1,0


In [ ]:
X = df2.drop("price", axis=1)
X


,width,height,area,lighting,installation,quantity,Acrylic,Aluminum,PVC,banner,channel_letter,monument,vinyl,high,low,medium
0,37,9,333,0,1,4,1,0,0,0,0,0,1,0,0,1
1,45,4,180,0,0,3,1,0,0,1,0,0,0,1,0,0
2,27,3,81,1,0,4,1,0,0,0,0,0,1,1,0,0
3,46,10,460,0,0,3,1,0,0,1,0,0,0,0,1,0
4,38,16,608,0,1,3,1,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1567,17,10,170,1,1,1,0,0,1,0,0,0,1,1,0,0
1568,41,4,164,0,0,2,0,0,1,1,0,0,0,1,0,0
1569,30,13,390,1,0,2,0,0,1,1,0,0,0,0,0,1
1570,28,10,280,1,1,2,0,0,1,0,0,0,1,0,1,0


In [ ]:
y = df2["price"]
y

0       21154.72
1        6277.33
2        6606.34
3       13612.43
4       47096.42
          ...   
1567     2269.83
1568     1945.99
1569     5039.92
1570     5238.19
1571     5270.49
Name: price, Length: 1572, dtype: float64

In [ ]:
X = df2.drop("price", axis=1)
y = df2.price

X.shape, y.shape

((1572, 16), (1572,))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10
)

In [ ]:
lr = LinearRegression()
lr.fit(X_train.values, y_train)

lr.score(X_test.values, y_test)

0.8316825690697796

In [ ]:
from sklearn.model_selection import cross_val_score, ShuffleSplit
from sklearn.linear_model import LinearRegression

cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

scores = cross_val_score(
    LinearRegression(),
    X.values,   # 🔥 critical fix
    y,
    cv=cv
)

scores

array([0.77070151, 0.80764133, 0.76029084, 0.80710187, 0.77990035])

In [ ]:
def find_best_model(X, y):
    from sklearn.linear_model import LinearRegression
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import GridSearchCV, ShuffleSplit
    import pandas as pd

    # 🔥 FIX: ensure consistent feature names (this was causing ALL your errors)
    X = X.copy()
    X.columns = X.columns.astype(str)

    algos = {
        "linear": {
            "model": LinearRegression(),
            "params": {}
        },
        "decision_tree": {
            "model": DecisionTreeRegressor(random_state=0),
            "params": {
                "criterion": ["squared_error", "friedman_mse"],
                "splitter": ["best", "random"]
            }
        },
        "random_forest": {
            "model": RandomForestRegressor(random_state=0),
            "params": {
                "n_estimators": [10, 50],
                "max_depth": [None, 10, 20]
            }
        }
    }

    scores = []

    cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

    for name, config in algos.items():
        gs = GridSearchCV(
            config["model"],
            config["params"],
            cv=cv,
            error_score="raise"
        )

        gs.fit(X.values, y)  # 🔥 critical fix (removes feature-name issues)

        scores.append({
            "model": name,
            "best_score": gs.best_score_,
            "best_params": gs.best_params_
        })

    return pd.DataFrame(scores)


# run it
find_best_model(X, y)

,model,best_score,best_params
0,linear,0.785127,{}
1,decision_tree,0.895483,"{'criterion': 'friedman_mse', 'splitter': 'best'}"
2,random_forest,0.948144,"{'max_depth': None, 'n_estimators': 50}"


In [ ]:
from sklearn.ensemble import RandomForestRegressor

final_model = RandomForestRegressor(
    max_depth=10,
    n_estimators=50,
    random_state=0
)

final_model.fit(X_train.values, y_train)

print(final_model.score(X_train.values, y_train))
print(final_model.score(X_test.values, y_test))

0.9903759140528487
0.9576845933034415


In [ ]:
def predict_price(model, width, height, sign_type, material,
                  complexity, lighting, installation, quantity):

    import pandas as pd

    area = width * height

    # create input row
    input_df = pd.DataFrame([{
        "width": width,
        "height": height,
        "area": area,
        "sign_type": sign_type,
        "material": material,
        "complexity": complexity,
        "lighting": lighting,
        "installation": installation,
        "quantity": quantity
    }])

    # one-hot encoding
    input_df = pd.get_dummies(input_df)

    # align columns with training data
    input_df = input_df.reindex(columns=X.columns, fill_value=0)

    # 🔥 CRITICAL FIX
    input_df.columns = input_df.columns.astype(str)

    # use numpy values (same as training)
    return model.predict(input_df.values)[0]

In [ ]:
predict_price(
    final_model,
    10, 5,
    "channel_letter",
    "Aluminum",
    "high",
    1,
    1,
    2
)

2978.604694054704

In [ ]:
predict_price(
    final_model,
    20, 10,
    "banner",
    "PVC",
    "low",
    0,
    0,
    1
)

4783.173865062387

In [ ]:
with open("sign_price_model.pickle", "wb") as f:
    pickle.dump(final_model, f)

In [ ]:
columns = {
    "data_columns": [col.lower() for col in X.columns]
}

with open("columns.json", "w") as f:
    f.write(json.dumps(columns))